<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module12/Lab6.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 6 — Closing the Hybrid Loop with COBYLA (Instructor)
**Quantum Optimization and Simulation — QAOA Laboratory Series**

Instructor version with answer key.

**Format:** 10–15 minute instructor walkthrough + about 45–60 minutes of independent work.

**Notebook style:** Most code is supplied. Cells marked **YOUR TURN** contain a small value, line, or function for you to complete.

> Qiskit displays measured bitstrings in the order `q_(n-1)...q_0`. When we discuss graph nodes, this notebook often converts them to `q_0...q_(n-1)` using `q0_first(...)`.

## Learning goals
- Close the hybrid quantum-classical loop.
- Use COBYLA to optimize \(\gamma,eta\).
- Plot convergence.
- Explore \(p=2\) and SPSA as optional extensions.

In [ ]:
# Run this once at the beginning of a fresh Google Colab session.
%pip -q install "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-optimization~=0.7" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
from qiskit_aer.primitives import SamplerV2

SEED = 123
SHOTS = 2048
sampler = SamplerV2(default_shots=SHOTS, seed=SEED)

def run_counts(qc, shots=SHOTS):
    "Run a measured circuit with Aer SamplerV2 and return counts."
    result = sampler.run([qc], shots=shots).result()
    return result[0].data.meas.get_counts()

def q0_first(qiskit_bitstring):
    "Convert Qiskit's displayed q_(n-1)...q_0 bitstring to q_0...q_(n-1)."
    return qiskit_bitstring.replace(" ", "")[::-1]

In [ ]:
from scipy.optimize import minimize

edges5 = [(0,3),(0,4),(1,3),(1,4),(2,3),(2,4)]

def cut_value_q0first(bitstring, edges):
    return sum(bitstring[i] != bitstring[j] for i, j in edges)

def expected_cut_from_counts(counts, edges):
    total = sum(counts.values())
    return sum(
        cut_value_q0first(q0_first(k), edges) * v / total
        for k, v in counts.items()
    )

def build_qaoa(gammas, betas):
    p = len(gammas)
    qc = QuantumCircuit(5)
    qc.h(range(5))

    for layer in range(p):
        gamma = gammas[layer]
        beta = betas[layer]

        for i, j in edges5:
            qc.cx(i, j)
            qc.rz(-gamma, j)
            qc.cx(i, j)

        for q in range(5):
            qc.rx(2 * beta, q)

    qc.measure_all()
    return qc

## Part A — Define the optimization objective

In [ ]:
history = []

def objective_p1(params):
    gamma, beta = params
    qc = build_qaoa([gamma], [beta])
    counts = run_counts(qc, shots=1024)
    avg_cut = expected_cut_from_counts(counts, edges5)
    history.append(avg_cut)
    return -avg_cut

## Part B — Run COBYLA

In [ ]:
history.clear()
initial_point = np.array([0.3, 0.6])

result = minimize(
    objective_p1,
    x0=initial_point,
    method="COBYLA",
    options={"maxiter": 40, "rhobeg": 0.5}
)

print(result)
print("Best gamma, beta =", result.x)
print("Best sampled average cut ≈", -result.fun)

plt.plot(history, marker=".")
plt.xlabel("Objective evaluation")
plt.ylabel("Average cut")
plt.title("COBYLA convergence")
plt.show()

**Expected:** the average cut should generally improve from the starting point, although shot noise makes the curve non-monotonic.

### YOUR TURN
Change the initial point to `[1.5, 1.5]`. Do you get the same final parameters? Do you get a similar final cut value?

## Optional extension A — p=2

In [ ]:
def objective_p2(params):
    gammas = params[:2]
    betas = params[2:]
    qc = build_qaoa(gammas, betas)
    counts = run_counts(qc, shots=1024)
    return -expected_cut_from_counts(counts, edges5)

# Uncomment to try.
# result_p2 = minimize(
#     objective_p2,
#     x0=np.array([0.5, 0.8, 0.3, 0.5]),
#     method="COBYLA",
#     options={"maxiter": 60}
# )
# print(result_p2.x, -result_p2.fun)

## Optional extension B — SPSA through built-in QAOA

In [ ]:
from qiskit_optimization.applications import Maxcut
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_optimization.minimum_eigensolvers import QAOA
from qiskit_optimization.optimizers import SPSA
from qiskit_aer.primitives import SamplerV2
import networkx as nx

# Optional:
# G = nx.Graph()
# G.add_nodes_from(range(5))
# G.add_edges_from(edges5)
# qp = Maxcut(G).to_quadratic_program()
# qaoa_spsa = QAOA(
#     sampler=SamplerV2(default_shots=1024, seed=SEED),
#     optimizer=SPSA(maxiter=60),
#     reps=1,
# )
# spsa_result = MinimumEigenOptimizer(qaoa_spsa).solve(qp)
# print(spsa_result.prettyprint())

## Questions
1. Why do we return `-avg_cut` to SciPy?
2. Why can two different parameter sets give similar solution quality?
3. Why might SPSA become attractive on noisy hardware?

## Instructor solutions

1. `scipy.optimize.minimize` minimizes a function, while Max-Cut is a maximization problem. Negating the average cut converts maximization into minimization.
2. QAOA landscapes are periodic and can have multiple local optima or parameter symmetries.
3. SPSA estimates a search direction using only a small number of noisy objective evaluations per iteration, making it attractive when each hardware evaluation is expensive/noisy.